| **Team Member**           | **Contributions**                                                                                                                                         |
|---------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Urjeet Parmar**         | - Prepared & cleaned dataset (tokenization, stopwords, lemmatization).<br>- Built initial NB model with one-hot (unigram+bigram) features.                |
| **Devendra Singh Sekhawat** | - Tuned NB models (GridSearchCV, alpha) and compared vectorizers.<br>- Integrated LIME/SHAP for local/global interpretability.                          |


    Negative Class: Precision = 0.81, Recall = 0.74, F1-score = 0.77
    Neutral Class: Precision = 0.38, Recall = 0.13, F1-score = 0.19
    Positive Class: Precision = 0.84, Recall = 0.95, F1-score = 0.89
    Overall Accuracy: 0.82
    Macro Average F1-score: 0.62
    Weighted Average F1-score: 0.79
    Best Cross-Validation Accuracy (Grid Search): 0.8200
    AUC Score: 0.8328

#1.Importing Necessary libraries

>



In [ ]:
pip install lime


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
import lime
from lime.lime_text import LimeTextExplainer
# Machine Learning & Evaluation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.pipeline import Pipeline

# WordCloud
from wordcloud import WordCloud


nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# Loaded Dataset and Dropped Null Values and it contains over 20k Reviews and ratings
df = pd.read_csv('data/yelp.csv').dropna()
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (10000, 2)


,text,stars
0,"I hate to say it. I really, really do.\nThe Ox...",3.0
1,Gray Fox Realty has demonstrated their ability...,5.0
2,Local here. Sorry to leave this review but the...,2.0
3,We make it a point to visit NOLA once or twice...,4.0
4,"Bomb.com \nThey have the most Delicious soups,...",5.0


#3. Map Star Ratings to Sentiment Classes

In [ ]:
#Mapped Ratings according to stars
def map_sentiment(stars):
    if stars >= 4:
        return 1  # Positive
    elif stars <= 2:
        return -1  # Negative
    else:
        return 0   # Neutral

df['sentiment'] = df['stars'].apply(map_sentiment)


#4. Preprocessing Function

In [ ]:
#Removed Stop Words
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    tokens = word_tokenize(text.lower())
    filtered_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalnum() and token not in stop_words
    ]
    return " ".join(filtered_tokens)

df['text'] = df['text'].apply(preprocess_text)


#5. Train-Test Split

In [ ]:
#Splitting the data into a way such that a test set contains more than 5000 rows
X = df['text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.26,
    stratify=y,
    random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])


Training set size: 7400
Test set size: 2600


#6. Building a Pipeline and Parameter Grid
6.1 Pipeline Explanation

    We want to try both CountVectorizer (with binary=True, ngram_range=(1,2)) and TfidfVectorizer (with the same n-gram range).
    We also want to try MultinomialNB, ComplementNB, and BernoulliNB.
    We’ll tune the alpha parameter for smoothing.

In [ ]:
#Defined a Pipeline to test on diffrent Naive Baiyes Models
pipeline = Pipeline([
    ('vect', CountVectorizer()),  # default, but we'll override in param_grid
    ('clf', MultinomialNB())
])


We’ll create a list of dictionaries, each specifying a combination of vectorizer and classifier we want to test:

In [ ]:
param_grid = [
    # ----- 1) CountVectorizer + MultinomialNB -----
    {
        'vect': [CountVectorizer(binary=True, ngram_range=(1,2))],
        'clf': [MultinomialNB()],
        'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    },
    # ----- 2) CountVectorizer + ComplementNB -----
    # {
    #     'vect': [CountVectorizer(binary=True, ngram_range=(1,2))],
    #     'clf': [ComplementNB()],
    #     'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    # },
    # # ----- 3) CountVectorizer + BernoulliNB -----
    # {
    #     'vect': [CountVectorizer(binary=True, ngram_range=(1,2))],
    #     'clf': [BernoulliNB()],
    #     'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    # },
    # # ----- 4) TfidfVectorizer + MultinomialNB -----
    # {
    #     'vect': [TfidfVectorizer(ngram_range=(1,2))],
    #     'clf': [MultinomialNB()],
    #     'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    # },
    # # ----- 5) TfidfVectorizer + ComplementNB -----
    # {
    #     'vect': [TfidfVectorizer(ngram_range=(1,2))],
    #     'clf': [ComplementNB()],
    #     'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    # },
    # # ----- 6) TfidfVectorizer + BernoulliNB -----
    # {
    #     'vect': [TfidfVectorizer(ngram_range=(1,2))],
    #     'clf': [BernoulliNB()],
    #     'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
    # }
]
